In [ ]:
!pip install langchain_groq langchain_community chromadb
from langchain_groq import ChatGroq
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import pandas as pd
import uuid
import chromadb

In [ ]:
llm = ChatGroq(
    temperature=0, 
    groq_api_key='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', 
    model_name="llama3-70b-8192"
)
response = llm.invoke("The first person to land on moon was ...")
print(response.content)

That's an easy one!

The first person to land on the moon was Neil Armstrong. He stepped out of the lunar module Eagle onto the moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to set foot on the moon.


In [9]:
import pandas as pd
df = pd.read_csv('350links.csv')
df

,Keywords,Links
0,Overview,https://docs.google.com/document/d/1p5rDNuK9mI...
1,Central Library,https://www.sust.edu/173/content-details/13/4
2,Achievement,https://www.sust.edu/112/achievements/49
3,"Contact, Numer",https://www.sust.edu/38/contact-us/49
4,"Schools, Institutes, Departments",https://sust.edu/144/schools-departments/105
5,Foreign Students,https://sust.edu/190/content-details/29/5
6,Residential Halls,https://sust.edu/178/content-details/8/2
7,"Students Organization, Clubs",https://sust.edu/181/student-organizations/109
8,Campus Life,https://sust.edu/175/content-details/7/2


In [ ]:
client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

for _, row in df.iterrows():
    collection.add(documents=row["Keywords"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [12]:
question = 'what are the department under applied science and technology school?'

In [ ]:
prompt_user_1 = PromptTemplate.from_template(
        """
        ### Question:
        {question}
        
        ### INSTRUCTION:
        Take the question above and make necessary changes to it so you can understand it better if
        needed.But do not give any answer. Just make changes to the question.Also do not add any
        preamble.
        ### QUESTION_ANSWER (NO PREAMBLE):
        
        """
        )

chain_question = prompt_user_1 | llm
res = chain_question.invoke({"question": str(question)})
print(res.content)
question = res.content

In [ ]:
links = collection.query(query_texts=question, n_results=1).get('metadatas', [])
link = links[0][0]['links']

loader = WebBaseLoader(str(link))
page_data = loader.load().pop().page_content
#print(page_data.strip())

In [ ]:
prompt_user = PromptTemplate.from_template(
        """
        ### Question:
        {question}
        
        ### INSTRUCTION:
        You are an assistant of an university. You will answer the questions given to you from the 
        provided information {info}.Use only the information which you are provided. Don't add
        any other information from your prior knowledge just from the links.
        Do not provide a preamble.Just only the required answer like a human.
        ### QUESTION_ANSWER (NO PREAMBLE):
        
        """
        )

chain_answer = prompt_user | llm
res = chain_answer.invoke({"question": str(question), "info": page_data.strip()})
print(res.content)